# Modelos de regressão com scikit-learn

Este notebook demonstra, **um modelo por seção**, como instanciar, treinar e avaliar regressores. Usamos `load_diabetes`, um dataset incluído no scikit-learn. A organização toma como referência *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (Aurélien Géron, 2ª ed.), capítulos 4 a 7.

## Preparação comum

O alvo é contínuo: uma medida quantitativa da progressão de diabetes após um ano. Não usaremos `Pipeline` nesta etapa. Quando houver escalonamento, o `MinMaxScaler` será ajustado somente no treino, prevenindo *data leakage*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option('display.float_format', lambda value: f'{value:.3f}')

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target

print(f'Observações: {X.shape[0]} | Features: {X.shape[1]}')
display(X.head())
display(y.describe())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X.columns, index=X_train.index
)

display(X_train_scaled.head())

In [ ]:
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X.columns, index=X_test.index
)

In [ ]:
resultados = []

def avaliar_modelo(nome, modelo, X_avaliacao):
    previsoes = modelo.predict(X_avaliacao)
    metricas = {
        'modelo': nome,
        'MAE': mean_absolute_error(y_test, previsoes),
        'RMSE': root_mean_squared_error(y_test, previsoes),
        'R²': r2_score(y_test, previsoes),
    }
    resultados.append(metricas)
    display(pd.DataFrame([metricas]))
    return previsoes

## Como a previsão vira métrica

A aplicação de cada modelo segue o mesmo fluxo: `fit(X_train, y_train)` aprende os parâmetros usando o treino; `predict(X_test)` gera $\hat{y}$ para features que o modelo não viu; as funções de métrica comparam $\hat{y}$ com os valores reais em `y_test`.

## 1. Baseline — `DummyRegressor`

A baseline prevê sempre a média do alvo de treino e ignora as features. Todo modelo posterior deve superá-la; caso contrário, ele não encontrou sinal útil nos dados.

### Hiperparâmetros

- `strategy`: estratégia usada para gerar a previsão constante. **Padrão:** `'mean'`. Exemplos: `'median'`, `'quantile'` e `'constant'`.
- `constant`: valor previsto quando `strategy='constant'`. **Padrão:** `None`. Exemplo: `constant=150`.
- `quantile`: quantil usado apenas quando `strategy='quantile'`. **Padrão:** `None`. Exemplos: `quantile=0.25` ou `quantile=0.75`.

In [ ]:
dummy = DummyRegressor()
dummy.fit(X_train, y_train)

In [ ]:
dummy_predicoes = avaliar_modelo('Dummy (média)', dummy, X_test)

In [ ]:
comparacao_dummy = pd.DataFrame({
    'valor_real (y_test)': y_test,
    'previsao (y_pred)': dummy_predicoes,
})
comparacao_dummy['residuo (real - previsto)'] = (
    comparacao_dummy['valor_real (y_test)'] - comparacao_dummy['previsao (y_pred)']
)
display(comparacao_dummy.head(10))

## 2. Regressão linear — `LinearRegression`

É o ponto de partida interpretável para relações aproximadamente lineares. Seus coeficientes mostram direção e intensidade da associação; ela é sensível a outliers e não captura curvaturas ou interações sem novas features.

### Hiperparâmetros

- `fit_intercept`: define se o modelo deve estimar o intercepto $\beta_0$. **Padrão:** `True`. Exemplo: `False` quando as features e o alvo já estão centrados.
- `positive`: quando ativado, restringe os coeficientes a valores não negativos. **Padrão:** `False`. Exemplo: `True` quando o domínio exige efeitos não negativos.
- `tol`: tolerância numérica usada pelos solvers aplicáveis para decidir quando encerrar o ajuste. **Padrão:** `1e-6`. Exemplos: `1e-4` (menos rigor) e `1e-8` (mais rigor).

In [ ]:
linear = LinearRegression()
linear.fit(X_train_scaled, y_train)

In [ ]:
linear_predicoes = avaliar_modelo('Linear Regression', linear, X_test_scaled)

In [ ]:
coeficientes = pd.Series(linear.coef_, index=X.columns).sort_values()
coeficientes.plot.barh(title='Coeficientes da regressão linear')
plt.xlabel('coeficiente')
plt.show()

## 3. Ridge — `Ridge`

Ridge adiciona penalização L2 aos coeficientes. É uma boa escolha quando há features correlacionadas e se deseja reduzir coeficientes instáveis. Requer escala comparável entre features; por isso usamos os dados com `MinMaxScaler`.

### Hiperparâmetros

- `alpha`: intensidade da penalidade L2. **Padrão:** `1.0`. Exemplos: `0.1`, `10.0` e `100.0`; valores maiores encolhem mais os coeficientes.
- `solver`: algoritmo numérico utilizado para encontrar os coeficientes. **Padrão:** `'auto'`. Exemplos: `'svd'`, `'cholesky'` e `'sag'`.
- `max_iter`: número máximo de iterações quando o solver é iterativo. **Padrão:** `None`. Exemplos: `1_000` e `10_000`.
- `tol`: tolerância que define a condição de convergência. **Padrão:** `1e-4`. Exemplos: `1e-3` e `1e-6`.
- `fit_intercept`: define se o intercepto deve ser estimado. **Padrão:** `True`. Exemplo: `False` para dados previamente centrados.

In [ ]:
ridge = Ridge()
ridge.fit(X_train_scaled, y_train)
ridge_predicoes = avaliar_modelo('Ridge', ridge, X_test_scaled)

## 4. Lasso — `Lasso`

Lasso usa penalização L1 e pode zerar coeficientes, funcionando também como seleção de features. É útil quando uma solução mais esparsa é desejada. Com features muito correlacionadas, pode manter uma e zerar outras de forma instável.

### Hiperparâmetros

- `alpha`: intensidade da penalidade L1. **Padrão:** `1.0`. Exemplos: `0.01`, `0.1` e `10.0`; valores maiores tendem a zerar mais coeficientes.
- `max_iter`: limite de iterações do otimizador. **Padrão:** `1_000`. Exemplos: `5_000` e `10_000`.
- `tol`: variação mínima aceita para considerar que o ajuste convergiu. **Padrão:** `1e-4`. Exemplos: `1e-3` e `1e-6`.
- `selection`: ordem de atualização dos coeficientes. **Padrão:** `'cyclic'`. Exemplo: `'random'` pode ser útil em problemas maiores.
- `fit_intercept`: define se o intercepto deve ser estimado. **Padrão:** `True`. Exemplo: `False` para dados previamente centrados.

In [ ]:
lasso = Lasso()
lasso.fit(X_train_scaled, y_train)

In [ ]:
lasso_predicoes = avaliar_modelo('Lasso', lasso, X_test_scaled)
print('Features com coeficiente diferente de zero:', (lasso.coef_ != 0).sum())

## 5. Elastic Net — `ElasticNet`

Elastic Net combina L1 e L2. É útil quando existem features correlacionadas, mas ainda se deseja regularização e possível seleção de variáveis.

### Hiperparâmetros

- `alpha`: intensidade total da regularização. **Padrão:** `1.0`. Exemplos: `0.01`, `0.1` e `10.0`.
- `l1_ratio`: proporção da penalidade L1; o restante corresponde à penalidade L2. **Padrão:** `0.5`. Exemplos: `0.1` (mais Ridge) e `0.9` (mais Lasso).
- `max_iter`: limite de iterações do otimizador. **Padrão:** `1_000`. Exemplos: `5_000` e `10_000`.
- `tol`: variação mínima aceita para considerar convergência. **Padrão:** `1e-4`. Exemplos: `1e-3` e `1e-6`.
- `selection`: ordem de atualização dos coeficientes. **Padrão:** `'cyclic'`. Exemplo: `'random'`.

In [ ]:
elastic_net = ElasticNet()
elastic_net.fit(X_train_scaled, y_train)
elastic_net_predicoes = avaliar_modelo('ElasticNet', elastic_net, X_test_scaled)

## 6. Árvore de decisão — `DecisionTreeRegressor`

Árvores aprendem regras e interações não lineares sem precisar de escalonamento. São mais tolerantes a outliers nas features, mas podem sobreajustar.

### Hiperparâmetros

- `criterion`: medida usada para avaliar a qualidade de uma divisão. **Padrão:** `'squared_error'`. Exemplos: `'friedman_mse'`, `'absolute_error'` e `'poisson'`.
- `max_depth`: profundidade máxima permitida para a árvore. **Padrão:** `None` (cresce até outros limites). Exemplos: `3`, `5` e `10`.
- `min_samples_split`: número mínimo de amostras necessário para dividir um nó. **Padrão:** `2`. Exemplos: `5`, `10` ou `0.05` (5% das amostras).
- `min_samples_leaf`: número mínimo de amostras que uma folha deve conter. **Padrão:** `1`. Exemplos: `2`, `5` ou `0.02`.
- `max_features`: quantidade de features consideradas a cada divisão. **Padrão:** `None` (todas). Exemplos: `'sqrt'`, `'log2'` ou `0.7`.
- `ccp_alpha`: intensidade da poda por complexidade. **Padrão:** `0.0`. Exemplos: `0.001` e `0.01`; valores maiores geram árvores menores.

In [ ]:
tree = DecisionTreeRegressor()
tree.fit(X_train, y_train)
tree_predicoes = avaliar_modelo('Decision Tree', tree, X_test)

## 7. Random Forest — `RandomForestRegressor`

Random Forest combina muitas árvores e reduz parte da variância de uma árvore isolada. É uma baseline forte para dados tabulares com não linearidades e interações. Não exige escala, mas não costuma extrapolar além da faixa de valores observada.

### Hiperparâmetros

- `n_estimators`: número de árvores na floresta. **Padrão:** `100`. Exemplos: `200`, `500` e `1_000`; mais árvores tendem a estabilizar o resultado, com maior custo computacional.
- `max_depth`: profundidade máxima de cada árvore. **Padrão:** `None`. Exemplos: `5`, `10` e `20`.
- `min_samples_split`: mínimo de amostras necessário para dividir um nó. **Padrão:** `2`. Exemplos: `5` e `10`.
- `min_samples_leaf`: mínimo de amostras em uma folha. **Padrão:** `1`. Exemplos: `2` e `5`; valores maiores ajudam a evitar folhas muito específicas.
- `max_features`: quantidade de features candidatas em cada divisão. **Padrão:** `1.0` (todas as features). Exemplos: `'sqrt'`, `0.5` e `0.8`.
- `bootstrap`: define se cada árvore é treinada com reamostragem com reposição. **Padrão:** `True`. Exemplo: `False`.
- `n_jobs`: número de processos usados para paralelizar o treinamento. **Padrão:** `None` (um processo). Exemplos: `-1` (todos os núcleos) e `4`.
- `random_state`: semente que permite reproduzir a aleatoriedade do ajuste. **Padrão:** `None`. Exemplos: `42` e `0`.

In [ ]:
random_forest = RandomForestRegressor()
random_forest.fit(X_train, y_train)
random_forest_predicoes = avaliar_modelo('Random Forest', random_forest, X_test)

## Comparação dos resultados

A tabela é construída com os resultados obtidos em cada seção. Ela não torna um modelo o melhor em todos os problemas: considere também estabilidade, interpretabilidade, custo e o significado do erro no domínio.

In [ ]:
resultados_df = pd.DataFrame(resultados).sort_values('RMSE').reset_index(drop=True)
display(resultados_df)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(resultados_df['modelo'], resultados_df['RMSE'])
ax.invert_yaxis()
ax.set(title='RMSE por modelo', xlabel='RMSE')
plt.show()

## Importância de features e SHAP

Em árvores, `feature_importances_` mede a redução de impureza, mas pode favorecer features contínuas ou com muitos valores. A importância por permutação mede a piora da métrica quando uma feature é embaralhada no teste. Ambas descrevem associações no modelo, não causalidade.

In [ ]:
importancia_arvore = pd.Series(random_forest.feature_importances_, index=X.columns).sort_values()
permutacao = permutation_importance(
    random_forest, X_test, y_test, scoring='neg_mean_absolute_error',
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)
importancia_permutacao = pd.Series(permutacao.importances_mean, index=X.columns).sort_values()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
importancia_arvore.plot.barh(ax=axes[0], title='Importância por impureza')
importancia_permutacao.plot.barh(ax=axes[1], title='Importância por permutação')
axes[0].set_xlabel('importância relativa')
axes[1].set_xlabel('piora média no MAE')
plt.tight_layout()
plt.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(random_forest)
shap_values = explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=X.columns)

Para aprofundar métricas, sensibilidade a escala e outliers, baseline e interpretação, consulte [`docs/001__REGRESSION.md`](../docs/001__REGRESSION.md).